# 函数

由于 OCaml 是一种函数式语言，所以关于函数有很多要学的内容。让我们开始吧。

```{important}
方法和函数不是同一个概念。方法是对象的一个组件，它隐式地有一个接收器，通常用关键字 `this` 或 `self` 访问。OCaml 的函数不是方法：它们不是对象的组件，也没有接收器。

有人可能会说所有方法都是函数，但不是所有函数都是方法。有些人甚至会进一步区分函数和过程。过程是指那些不返回有意义值的函数，比如 Java 中的 `void` 返回类型或 Python 中的 `None` 返回值。

所以如果你来自面向对象的背景，请注意术语。**这里的一切严格来说都是函数，不是方法。**
```

## 函数定义

{{ video_embed | replace("%%VID%%", "vCxIlagS7kA")}}

下面的代码
```ocaml
let x = 42
```
其中有一个表达式 (`42`)，但它本身不是表达式。相反，它是一个
*定义*。定义会把值绑定到名称；在这个例子中，值 `42`
被绑定到名称 `x`。OCaml 手册描述了
[定义][definitions]（参见该页面上第三个主要分组，标题为“*definition*”），
但该手册页主要还是供参考，而不是用来学习。
定义不是表达式，表达式也不是定义 &mdash; 它们是不同的语法类别。

[definitions]: https://ocaml.org/manual/modules.html

现在，让我们关注一种特定类型的定义，*函数
定义*。非递归函数的定义如下：

```ocaml
let f x = ...
```

{{ video_embed | replace("%%VID%%", "_x82qitu2R8")}}

递归函数的定义如下：

```ocaml
let rec f x = ...
```

区别只是 `rec` 关键字。这可能有点令人惊讶
你必须明确地添加一个关键字来使函数递归，因为大多数
语言默认认为它们是。 OCaml 没有做出这样的假设，
不过。 （Scheme 语言家族也没有。）

最著名的递归函数之一是阶乘函数。在 OCaml 中，
可以写成如下：

In [ ]:
(** [fact n] is [n!].
    Requires: [n >= 0]. *)
let rec fact n = if n = 0 then 1 else n * fact (n - 1)

我们在函数上方提供了规范注释来记录
函数的前置条件 (`Requires`) 和后置条件 (`is`)。

请注意，和许多语言一样，OCaml 整数不是“数学意义上”的整数，
而是受固定比特数限制。[手册][man]规定，（有符号）整数至少有
31 位，但也可能更宽。随着体系结构的发展，这个大小也在增长。
在当前实现中，OCaml 整数是 63 位。因此，如果你用足够大的输入做测试，
可能会开始看到奇怪的结果。问题出在机器算术，而不是 OCaml。
（感兴趣的读者可能会问：为什么是 31 或 63，而不是 32 或 64？
OCaml 垃圾收集器需要区分整数和指针。因此，它们的运行时表示会借用一位，
用来标记一个机器字是整数还是指针。）

[man]: https://ocaml.org/manual/values.html#sss:values:integer


这是另一个递归函数：

In [ ]:
(** [pow x y] is [x] to the power of [y].
     Requires: [y >= 0]. *)
let rec pow x y = if y = 0 then 1 else x * pow x (y - 1)

请注意，我们不必在这两个函数中写出任何类型：OCaml
编译器会自动为我们推断它们。编译器用算法解决这个*类型推断*问题，
但我们也可以自己推断。它就像一个可以靠推理能力解开的谜题：

* 由于 `if` 表达式可以在 `then` 分支中返回 `1`，根据 `if` 的类型规则，
  整个 `if` 表达式的类型是 `int`。

* 由于 `if` 表达式的类型是 `int`，该函数的返回类型也必须是 `int`。

* 由于 `y` 使用相等运算符与 `0` 比较，`y` 必须是 `int`。

* 由于 `x` 使用 `*` 运算符与另一个表达式相乘，`x` 必须是 `int`。

如果我们出于某种原因想写下类型，我们可以这样做：
```ocaml
let rec pow (x : int) (y : int) : int = ...
```
当我们为 `x` 编写*类型注释*时，括号是必需的
`y`。我们通常会省略这些注释，因为这样更简单
编译器推断它们。有时你会想要明确地
写下类型。一个特别有用的时间是当你从以下位置收到类型错误时
你不明白的编译器。显式注释类型会有所帮助
调试此类错误消息。

**语法。**
函数定义的语法：
```ocaml
let rec f x1 x2 ... xn = e
```
`f` 是一个元变量，指示用作函数的标识符
名字。这些标识符必须以小写字母开头。剩余的
可以在手册中找到[小写标识符的规则][lowercase]。
名称 `x1` 到 `xn` 是指示参数标识符的元变量。这些
遵循与函数标识符相同的规则。如果出现以下情况，则需要关键字 `rec`
`f` 是一个递归函数；否则可以省略。

[lowercase]: https://ocaml.org/manual/lex.html#lowercase-ident

请注意，函数定义的语法实际上比
OCaml 真正允许的。我们将详细了解一些增强语法
在接下来的几周内定义函数。但就目前而言，这简化了
版本将帮助我们集中注意力。

可以使用 `and` 关键字定义相互递归函数：
```ocaml
let rec f x1 ... xn = e1
and g y1 ... yn = e2
```

例如：

In [ ]:
(** [even n] is whether [n] is even.
    Requires: [n >= 0]. *)
let rec even n =
  n = 0 || odd (n - 1)

(** [odd n] is whether [n] is odd.
    Requires: [n >= 0]. *)
and odd n =
  n <> 0 && even (n - 1);;

{{ video_embed | replace("%%VID%%", "W0rO84YXIXo")}}

函数类型的语法是：
```ocaml
t -> u
t1 -> t2 -> u
t1 -> ... -> tn -> u
```
`t` 和 `u` 是指示类型的元变量。类型 `t -> u` 是以下类型
接受 `t` 类型的输入并返回 `u` 类型的输出的函数。我们
可以将 `t1 -> t2 -> u` 视为采用两个输入的函数类型，
第一个类型为 `t1` ，第二个类型为 `t2` ，并返回输出
输入 `u`。对于采用 `n` 参数的函数也是如此。

**动态语义。** 函数定义没有动态语义。
没有什么可以评价的。 OCaml 只是记录名称 `f` 被绑定
具有给定参数 `x1..xn` 和给定主体 `e` 的函数。仅
以后应用这个函数的时候，会不会做一些评估呢？

**静态语义。** 函数定义的静态语义：

* 对于非递归函数：如果假设 `x1 : t1` 和 `x2 : t2` 和 ...
和 `xn : tn`，我们可以得出 `e : u` 的结论，那么
  `f : t1 -> t2 -> ... -> tn -> u`。
* 对于递归函数：如果假设 `x1 : t1` 和 `x2 : t2` 以及 ...
和 `xn : tn` 和 `f : t1 -> t2 -> ... -> tn -> u`，我们可以得出结论：
  `e : u`，然后`f : t1 -> t2 -> ... -> tn -> u`。

请注意递归函数的类型检查规则如何假设
函数标识符 `f` 具有特定类型，然后检查是否
在该假设下，函数体的类型是正确的。这是因为 `f` 是
在函数体本身的范围内（就像参数在范围内一样）。

## 匿名函数

{{ video_embed | replace("%%VID%%", "JwoIIrj0bcM")}}

我们已经知道我们可以拥有不与名称绑定的值。
例如，可以在顶层输入整数 `42`，而无需
给它一个名字：

In [ ]:
42

或者我们可以将它绑定到一个名称：

In [ ]:
let x = 42

同样，OCaml 函数不必有名称；他们可能是*匿名*。
例如，下面是一个增加其输入的匿名函数：
`fun x -> x + 1`。这里，`fun`是一个关键字，表示匿名函数，`x`
是参数，`->` 将参数与主体分开。

现在我们有两种编写增量函数的方法：

In [ ]:
let inc x = x + 1
let inc = fun x -> x + 1

它们在语法上不同，但在语义上等效。也就是说，即使
尽管它们涉及不同的关键字并将一些标识符放在不同的位置
地方，它们的意思是一样的。

{{ video_embed | replace("%%VID%%", "zHHCD7MOjmw")}}

匿名函数也称为 *lambda 表达式*，该术语来自
*lambda 演算*，它是相同计算的数学模型
感觉图灵机是一种计算模型。在 lambda 演算中，
`fun x -> e` 将写为 $\lambda x . e$。 $\lambda$ 表示
匿名函数。

现在似乎有点神秘，为什么我们需要这样的函数
没有名字。不用担心;我们稍后会在课程中看到它们的良好用途，
尤其是当我们学习所谓的“高阶编程”时。特别是，我们
通常会创建匿名函数并将它们作为输入传递给其他函数。

**语法。**
```ocaml
fun x1 ... xn -> e
```

**静态语义。**

* 如果假设
`x1 : t1` 和 `x2 : t2` 以及 ... 和 `xn : tn`，我们可以得出结论 `e : u`，
  然后`fun x1 ... xn -> e : t1 -> t2 -> ... -> tn -> u`。

**动态语义。** 匿名函数已经是一个值。没有
要执行的计算。

## 函数应用

{{ video_embed | replace("%%VID%%", "fgCTDhXAYnQ")}}

与以下内容相比，这里我们介绍了函数应用程序的稍微简化的语法
OCaml 实际允许的内容。

**语法。**
```ocaml
e0 e1 e2 ... en
```
第一个表达式 `e0` 是函数，它应用于参数 `e1`
通过`en`。请注意，参数周围不需要括号
指示函数应用程序，因为它们是 C 系列语言中的，
包括Java。

**静态语义。**

* 如果 `e0 : t1 -> ... -> tn -> u` 和 `e1 : t1` 和 ... 和 `en : tn`
然后`e0 e1 ... en : u`。

**动态语义。**

要评估 `e0 e1 ... en`：

1. 将 `e0` 评估为函数。同时评估参数表达式 `e1`
从 `en` 到值 `v1` 到 `vn`。

对于 `e0`，结果可能是一个匿名函数 `fun x1 ... xn ->
   e` or a name `f`. In the latter case, we need to find the definition of `f`,
   我们可以假设其形式为 `let rec f x1 ... xn =
   e`.  Either way, we now know the argument names `x1` through `xn` 和
   主体`e`。

2. 将每个值 `vi` 替换为相应的参数名称 `xi`
函数体 `e` 。该替换会产生新的表达式 `e'`。

3. 将 `e'` 评估为值 `v`，这是评估的结果
`e0 e1 ... en`。

如果将这些评估规则与 `let` 表达式的规则进行比较，你
会注意到它们都涉及替换。这并非偶然。事实上，
程序中出现 `let x = e1 in e2` 的任何地方，我们都可以将其替换为
`(fun x -> e2) e1`。它们在语法上不同，但在语义上不同
等价。本质上，`let` 表达式只是匿名的语法糖
函数应用。

## 管道

{{ video_embed | replace("%%VID%%", "arS9kEqCFEU")}}

OCaml 中有一个内置的中缀运算符用于函数应用，称为
*管道*运算符，写作 `|>`。想象一下，描绘一个三角形指向
到右边。这个比喻是，值通过管道发送
从左到右。例如，假设我们有增量函数 `inc` 来自
上面还有一个函数 `square` 对其输入进行平方：

In [ ]:
let square x = x * x

以下是对 `6` 进行平方的两种等效方法：

In [ ]:
square (inc 5);;
5 |> inc |> square;;

后者使用管道运算符通过`inc`函数发送`5`，
然后通过 `square` 函数发送结果。这是一个不错的，
在 OCaml 中表达计算的惯用方式。前一种方式可以说是
不太优雅：它涉及编写额外的括号并需要读者的
眼睛会跳来跳去，而不是从左到右线性移动。后者
当应用的函数数量增加时，方式会很好地扩展，而
前一种方式需要越来越多的括号：

In [ ]:
5 |> inc |> square |> inc |> inc |> square;;
square (inc (inc (square (inc 5))));;

一开始可能会感觉很奇怪，但是尝试在你自己的中使用管道运算符
下次当你发现自己正在编写一大串函数时，请编写代码
应用程序。

由于 `e1 |> e2` 只是 `e2 e1` 的另一种写法，所以我们不需要声明
`|>` 的语义：它与函数应用程序相同。这两个
程序是语法不同但是表达式的另一个例子
语义上等价。

## 多态函数

{{ video_embed | replace("%%VID%%", "UWmxYBEKzN8")}}

*identity* 函数是简单返回其输入的函数：

In [ ]:
let id x = x

或者等效地作为匿名函数：

In [ ]:
let id = fun x -> x

`'a` 是一个*类型变量*：它代表未知类型，就像
常规变量代表未知值。类型变量总是以 a 开头
单引号。常用的类型变量包括`'a`、`'b`和`'c`，其中
OCaml 程序员通常用希腊语发音：alpha、beta 和 gamma。

我们可以将恒等函数应用于我们喜欢的任何类型的值：

In [ ]:
id 42;;
id true;;
id "bigred";;

因为你可以将 `id` 应用于多种类型的值，所以它是一个*多态*
函数：它可以应用于多种（*poly*）形式（*morph*）。

通过手动类型注释，可以给出更具限制性的类型
多态函数而不是编译器自动推断的类型。
例如：

In [ ]:
let id_int (x : int) : int = x

除了两个手动类型注释之外，该函数与 `id` 相同。
因此，我们不能像 `id` 那样将 `id_int` 应用于 `bool`：

In [ ]:
id_int true

`id_int` 的另一种书写方式是 `id`：

In [ ]:
let id_int' : int -> int = id

实际上，我们采用了 `'a -> 'a` 类型的值，并将其绑定到一个名称，该名称的名称
类型被手动指定为 `int -> int`。你可能会问，为什么会这样
工作？毕竟他们不是同一类型。

考虑这个问题的一种方法是从**行为的角度来看。** `id_int` 的类型
指定其行为的一方面：给定 `int` 作为输入，它承诺
产生 `int` 作为输出。事实证明 `id` 也做出了同样的承诺：
给定 `int` 作为输入，它也会返回 `int` 作为输出。现在 `id` 也
做出更多承诺，例如：给定 `bool` 作为输入，它将返回
`bool` 作为输出。因此，通过将 `id` 绑定到更严格的类型 `int -> int`，我们
我们只是丢弃了所有这些与当前需求无关的额外承诺。
当然，这会丢失信息，但至少不会违背承诺。
因此，当我们需要一个 `int -> int` 类型的函数时，
使用 `'a -> 'a` 类型的函数总是安全的。

反之则不然。如果我们需要一个 `'a -> 'a` 类型的函数但尝试过
使用 `int -> int` 类型的函数，一旦有人
传递了另一种类型的输入，例如 `bool`。为了防止这种麻烦，OCaml
使用以下代码做了一些可能令人惊讶的事情：

In [ ]:
let id' : 'a -> 'a = fun x -> x + 1

函数 `id'` 实际上是增量函数，而不是恒等函数。所以
向其传递 `bool` 或 `string` 或一些复杂的数据结构是不安全的；
`+` 唯一可以安全操作的数据是整数。因此 OCaml
*实例化*类型变量 `'a` 到 `int`，从而阻止我们
将 `id'` 应用于非整数：

In [ ]:
id' true

这引导我们采用另一种更机械的方式来思考这一切
**应用**的角度。这里指的是函数如何
应用于参数：当求值应用 `id 5` 时，参数
`x` *实例化*值为 `5`。同样，`id` 类型中的 `'a` 是
在该应用程序中使用类型 `int` 进行实例化。所以如果我们写

In [ ]:
let id_int' : int -> int = id

事实上，我们正在用 `int` 类型实例化 `id` 类型中的 `'a` 。
就像没有办法“取消应用”函数一样。例如，给定 `5`，
我们无法倒推出 `id 5`；同样，我们也无法取消这次类型
实例化并将 `int` 更改回 `'a`。

为了准确起见，假设我们有一个 `let` 定义[或表达式]：

```ocaml
let x = e [in e']
```

并且 OCaml 推断 `x` 具有类型 `t`，其中包括一些类型变量 `'a`，
`'b` 等。然后我们就可以实例化这些类型变量。我们可以做
通过将函数应用于揭示类型的参数
实例化应该是（如 `id 5` 中）或通过类型注释（如
`id_int'`)，等等。但我们必须与
实例化。例如，我们不能采用 `'a -> 'b -> 'a` 类型的函数
并在类型 `int -> 'b -> string` 处实例化它，因为实例化
`'a` 在它发生的两个地方都不是同一类型：

In [ ]:
let first x y = x;;
let first_int : int -> 'b -> int = first;;
let bad_first : int -> 'b -> string = first;;

## 带标签和可选参数

函数的类型和名称通常可以让你很好地了解函数的
参数应该是什么。但是，对于具有多个参数的函数（尤其是
相同类型的参数），给它们贴上标签会很有用。例如，你
可能会猜测函数 `String.sub` 返回给定的子字符串
字符串（你是对的）。你可以输入 `String.sub` 来查找它
类型：

In [ ]:
String.sub;;

但从类型上并不清楚如何使用&mdash;你被迫咨询
文档。

OCaml 支持函数的标记参数。你可以声明这种
使用以下语法的函数：

In [ ]:
let f ~name1:arg1 ~name2:arg2 = arg1 + arg2;;

可以通过按任一顺序传递带标签的参数来调用此函数：

```ocaml
f ~name2:3 ~name1:4
```

参数的标签通常与其变量名称相同。OCaml
提供了这种情况的简写。以下是等效的：

```ocaml
let f ~name1:name1 ~name2:name2 = name1 + name2
let f ~name1 ~name2 = name1 + name2
```

是否使用带标签的参数很大程度上取决于品味。它们传达了额外的
信息，但它们也会给类型带来混乱。

为其编写带标签参数和显式类型注释的语法
它是：

```
let f ~name1:(arg1 : int) ~name2:(arg2 : int) = arg1 + arg2
```

也可以使一些参数可选。当调用时没有
可选参数，将提供默认值。要声明这样一个函数，
使用以下语法：

In [ ]:
let f ?name:(arg1=8) arg2 = arg1 + arg2

然后你可以应用带或不带参数的函数：

In [ ]:
f ~name:2 7

In [ ]:
f 7

## 部分应用

{{ video_embed | replace("%%VID%%", "85xVK0wmDTw")}}

我们可以定义一个加法函数如下：

In [ ]:
let add x y = x + y

这是一个相当相似的函数：

In [ ]:
let addx x = fun y -> x + y

函数 `addx` 将整数 `x` 作为输入并返回类型为 *function* 的函数
`int -> int` 会将 `x` 添加到传递给它的任何内容中。

`addx` 的类型是 `int -> int -> int`。 `add` 的类型也是
`int -> int -> int`。所以从它们的类型来看，它们是相同的
函数。但 `addx` 的形式暗示了一些有趣的事情：我们可以应用它
一个参数。

In [ ]:
let add5 = addx 5

In [ ]:
add5 2

事实证明，使用 `add` 也可以完成同样的事情：

In [ ]:
let add5 = add 5

In [ ]:
add5 2;;

我们刚刚所做的称为*部分应用*：我们部分应用了
函数 `add` 到一个参数，即使你通常认为它是一个
多参数函数。这是有效的，因为以下三个函数是
*语法不同*但*语义等价*。也就是说，它们是
表达相同计算的不同方式：

```ocaml
let add x y = x + y
let add x = fun y -> x + y
let add = fun x -> (fun y -> x + y)
```

所以 `add` 实际上是一个接受参数 `x` 并返回一个函数的函数
`(fun y -> x + y)`。这引导我们发现一个深刻的真理......

## 函数结合性

你准备好接受真相了吗？  深吸一口气。  这里...

**每个 OCaml 函数只接受一个参数。**

为什么？考虑 `add`：虽然我们可以将其写为 `let add x y = x + y`，但我们知道
这在语义上等同于 `let add = fun x -> (fun y -> x + y)`。并且在
一般，

```ocaml
let f x1 x2 ... xn = e
```

在语义上等价于

```ocaml
let f =
  fun x1 ->
    (fun x2 ->
       (...
          (fun xn -> e)...))
```

因此，即使你将 `f` 视为采用 `n` 参数的函数，
实际上，它是一个带有 1 个参数并返回一个函数的函数。

这种函数的类型

```ocaml
t1 -> t2 -> t3 -> t4
```

真正的意思是一样的

```ocaml
t1 -> (t2 -> (t3 -> t4))
```

也就是说，函数类型是*右关联的*：有隐式括号
围绕函数类型，从右到左。这里的直觉是一个函数
接受一个参数并返回一个新函数，该函数期望剩余的
论据。

另一方面，函数应用是*左关联*：
是函数应用程序周围的隐式括号，从左到右。
所以

```ocaml
e1 e2 e3 e4
```

真正的意思是一样的

```ocaml
((e1 e2) e3) e4
```

这里的直觉是最左边的表达式抓住下一个
其右侧的表达式作为其单个参数。

## 作为函数的运算符

{{ video_embed | replace("%%VID%%", "OVKOx8UiwxM")}}

加法运算符 `+` 的类型为 `int -> int -> int`。一般是这样写的
*中缀*，例如 `3 + 4`。通过在它周围加上括号，我们可以将其变成
*前缀*运算符：

In [ ]:
( + )

In [ ]:
( + ) 3 4;;

In [ ]:
let add3 = ( + ) 3

In [ ]:
add3 2

相同的技术适用于任何内置运算符。

通常这些空格是不必要的。我们可以写 `(+)` 或 `( + )`，但它是
最好包括它们。当心乘法，它*必须*写成
`( * )`，因为 `(*)` 将被解析为注释的开始。

我们甚至可以定义自己的新中缀运算符，例如：
```ocaml
let ( ^^ ) x y = max x y
```
现在 `2 ^^ 3` 的计算结果为 `3`。

标点符号可用于创建中缀运算符的规则不是
必然是直观的。这些运算符的相对优先级也不是
将被解析。所以要小心这种用法。

## 尾递归

考虑以下看似无趣的函数，它从 1 计数到
`n`：

In [ ]:
(** [count n] is [n], computed by adding 1 to itself [n] times.  That is,
    this function counts up from 1 to [n]. *)
let rec count n =
  if n = 0 then 0 else 1 + count (n - 1)

数到 10 没有问题：

In [ ]:
count 10

数到 100,000 也没有问题：

In [ ]:
count 100_000

但尝试数到 1,000,000 时，你会收到以下错误：
```
Stack overflow during evaluation (looping recursion?).
```

这是怎么回事？

**调用堆栈。**问题是*调用堆栈*的大小有限。你
可能在你的一门入门编程课程中学到了最
语言通过堆栈实现函数调用。该堆栈包含一个元素
对于每个已开始但尚未完成的函数调用。每个
元素存储诸如局部变量值之类的信息
当前正在执行函数中的指令。当评估
一个函数体调用另一个函数，在调用时推送一个新元素
堆栈，并在被调用函数完成时弹出。

堆栈的大小通常受到操作系统的限制。所以如果
堆栈空间不足，无法进行另一个函数调用。
通常这不会发生，因为没有理由做那么多
返回之前连续的函数调用。如果确实发生这种情况，
操作系统有充分的理由让该程序停止：它可能
正在耗尽整个系统上*所有*可用的内存
计算机，从而损害同一计算机上运行的其他程序。 `count`
函数不太可能做到这一点，但这个函数会：

In [ ]:
let rec count_forever n = 1 + count_forever n

所以操作系统为了安全起见限制了调用栈的大小。这意味着
最终 `count` 将在足够大的输入上耗尽堆栈空间。通知
这种选择如何真正独立于编程语言。所以这个也一样
问题可能并且确实发生在 OCaml 以外的语言中，包括 Python 和
Java。你只是不太可能看到它在那里显现，因为你
可能从来没有用这些语言编写过那么多的递归函数。

**尾递归。** 这个问题有一个解决方案，在
[Guy Steele 1977 年关于 LISP 的论文][lisp-tailcall]。解决方案，*尾调用
优化*，需要程序员和开发人员之间的一些合作
编译器。程序员对该函数进行了一些重写，
然后编译器会注意到并应用优化。让我们看看它是如何工作的。

[lisp-tailcall]: https://dl.acm.org/doi/pdf/10.1145/800179.810196

假设递归函数 `f` 调用自身然后返回结果
那个递归调用。我们的 `count` 函数*不*这样做：

In [ ]:
let rec count n =
  if n = 0 then 0 else 1 + count (n - 1)

相反，在递归调用 `count (n - 1)` 之后，进行计算
剩余：计算机仍然需要将 `1` 添加到该调用的结果中。

但我们作为程序员可以重写 `count` 函数，使其*不*
递归调用后需要进行任何额外的计算。诀窍是
创建带有额外参数的辅助函数：

In [ ]:
let rec count_aux n acc =
  if n = 0 then acc else count_aux (n - 1) (acc + 1)

let count_tr n = count_aux n 0

函数 `count_aux` 与我们原来的 `count` 几乎相同，但它增加了一个
名为 `acc` 的额外参数，这是惯用的，代表“累加器”。
这个想法是我们想要从函数返回的值是缓慢的，
每个递归调用，都在其中累积。 “剩余计算量”
&mdash;添加 1&mdash; 现在发生在递归调用*之前*
*之后*。  当递归的基本情况最终到达时，函数
现在返回 `acc`，其中已累积答案。

但原始的基本情况 0 仍然需要存在于代码中的某个地方。
确实如此，因为 `acc` 的原始值被传递给 `count_aux`。
现在 `count_tr` （我们很快就会明白为什么名字是“tr”）可以工作了
作为我们原来的 `count` 的替代品。

至此我们就完成了程序员的职责，但是大概
不清楚我们为什么要做出这样的努力。毕竟 `count_aux` 仍然会调用
本身像 `count` 那样递归了太多次，最终溢出了
堆栈。

这就是编译器的责任发挥作用的地方。一个好的编译器（以及
OCaml 编译器这样就很好）可以注意到 *tail 中何时有递归调用
位置*，这是一种技术方式，表示“不再需要计算”
返回后完成”。对 `count_aux` 的递归调用位于尾部位置；
对 `count` 的递归调用不是。它们再次出现，以便你进行比较
他们：

In [ ]:
let rec count n =
  if n = 0 then 0 else 1 + count (n - 1)

let rec count_aux n acc =
  if n = 0 then acc else count_aux (n - 1) (acc + 1)

这就是尾部位置很重要的原因： **尾部位置的递归调用并不重要
需要一个新的堆栈框架。它可以只重用现有的堆栈框架。**那就是
因为现有的堆栈框架中没有任何东西可以使用了！没有
计算尚未完成，因此没有局部变量或下一条指令
执行等不再重要。这些内存都不需要被读取
再次，因为该调用实际上已经完成。所以与其浪费
通过分配另一个堆栈帧的空间，编译器“回收”使用的空间
通过前一帧。

这就是*尾调用优化*。它甚至可以应用于超出范围的情况
递归函数（如果调用函数的堆栈帧适当兼容）
与被叫方。而且，这是一件大事。尾调用优化减少了
堆栈空间要求从线性到恒定。而 `count` 需要 $O(n)$
堆栈帧，`count_aux` 只需要 $O(1)$，因为相同的帧会被重用
每次递归调用都会一遍又一遍。这实际上意味着 `count_tr`
可以数到 1,000,000：

In [ ]:
count_tr 1_000_000

最后，为什么我们将这个函数命名为`count_tr`？ “tr”代表*tail
递归*。尾递归函数是一种递归函数，其递归函数
调用全部位于尾部位置。换句话说，它是一个函数（除非
还有其他病症）不会耗尽堆栈。

**尾递归的重要性。**有时是函数式程序员的新手
过于关注它。如果你只关心写初稿
对于函数来说，你可能不需要担心尾递归。这是
如果需要的话，稍后可以很容易地使其尾递归，只需添加一个
累加器参数。或者也许你应该重新思考你是如何设计的
函数。以 `count` 为例：这有点愚蠢。但稍后我们会看到
一些并不愚蠢的例子，例如迭代包含数千个的列表
元素。

编译器支持优化非常重要。否则，
作为程序员，你对代码所做的转换没有任何区别。确实，
大多数编译器都支持它，至少作为一个选项。 Java 是一个值得注意的
例外。

**尾递归的秘诀。** 简而言之，这就是我们如何创建一个函数
是尾递归：

1. 将函数更改为辅助函数。添加一个额外的参数：
累加器，通常命名为 `acc`。
1. 编写调用帮助器的函数的新“主”版本。它通过了
原始基本情况的返回值作为初始值
   累加器。
1. 更改辅助函数以返回基本情况下的累加器。
1. 更改辅助函数的递归大小写。它现在需要做额外的事情
在递归调用之前处理累加器参数。这是唯一的
   这一步需要很多独创性。

**示例：阶乘。** 让我们将此阶乘函数转换为
尾递归：

In [ ]:
(** [fact n] is [n] factorial. *)
let rec fact n =
  if n = 0 then 1 else n * fact (n - 1)

首先，我们更改它的名称并添加一个累加器参数：
```ocaml
let rec fact_aux n acc = ...
```

其次，我们编写一个新的“main”函数，用原始函数调用助手
作为累加器的基本情况：
```ocaml
let fact_tr n = fact_aux n 1
```

第三，我们更改辅助函数以在基本情况下返回累加器：
```ocaml
if n = 0 then acc ...
```

最后，我们改变递归的情况：
```ocaml
else fact_aux (n - 1) (n * acc)
```

把它们放在一起，我们有：

In [ ]:
let rec fact_aux n acc =
  if n = 0 then acc else fact_aux (n - 1) (n * acc)

let fact_tr n = fact_aux n 1

这是一个很好的练习，但也许不值得。  甚至在我们用尽之前
堆栈空间，计算会出现整数溢出：

In [ ]:
fact 50

为了解决这个问题，我们求助于 OCaml 的大整数库，
[Zarith][zarith]。这里我们使用了一些超越一切的 OCaml 功能
到目前为止我们已经看到了，但希望没有什么特别令人惊讶的。 （如果你想
按照此代码，首先在 OPAM 中安装 Zarith
`opam install zarith`。）

[zarith]: https://antoinemine.github.io/Zarith/doc/latest/Z.html

In [ ]:
#use "topfind";;

In [ ]:
#require "zarith.top";;

In [ ]:
let rec zfact_aux n acc =
  if Z.equal n Z.zero then acc else zfact_aux (Z.pred n) (Z.mul acc n);;

let zfact_tr n = zfact_aux n Z.one;;

zfact_tr (Z.of_int 50)

如果你愿意，可以使用该代码来计算 `zfact_tr 1_000_000` 而不需要堆栈
或整数溢出，尽管这需要几分钟的时间。

模块章节将详细解释我们上面使用的 OCaml 功能，
但现在：

- `#require` 加载该库，该库提供了一个名为 `Z` 的模块。回想一下
$\mathbb{Z}$ 是数学中用来表示整数的符号。

- `Z.n` 表示在 `Z` 内部定义的名称 `n`。

- 类型 `Z.t` 是大整数类型的库名称。

- 我们使用库值 `Z.equal` 进行相等比较，使用 `Z.zero` 表示 0，
`Z.pred` 表示前驱（即减 1）， `Z.mul` 表示乘法，
  `Z.one` 表示 1，`Z.of_int` 将原始整数转换为大整数。